# Analyse: Farbsättigung


In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
from tqdm import tqdm


In [3]:
DATA_DIR = Path('../../data')

COMBINED_CSV = DATA_DIR / '03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUTPUT_CSV = DATA_DIR / '04_analysis_results' / 'visual_features' / '02_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_saturation.csv'
FRAME_ROOT = DATA_DIR / '02_media/stratified_sample/frames'  # one folder with per-video subfolders
SOURCE_FILTER = None  # e.g., ['ai', 'real'] or None to use all
DEFAULT_MAX_FRAMES_PER_VIDEO = 60 

device = torch.device('mps') if torch.backends.mps.is_available() else 'cpu'
print(f'Running on device: {device}')
df = pd.read_csv(COMBINED_CSV)


Running on device: cpu


In [ ]:
if SOURCE_FILTER:
    df = df[df['influencer_type'].isin(SOURCE_FILTER)].copy()

video_id_col = 'video_id' if 'video_id' in df.columns else 'id'


def has_frames(video_id: str) -> bool:
    return (FRAME_ROOT / video_id).is_dir()


# Frame-Ordner erneut prüfen
video_ids = df[video_id_col].astype(str)
df['has_frames'] = [has_frames(video_id) for video_id in video_ids]

missing_ids = video_ids[~df['has_frames']].tolist()
print(f'Total videos in CSV: {len(df)}')
print(f'Videos with frame folder: {df["has_frames"].sum()}')
print(f'Videos missing frame folder: {len(missing_ids)}')

if missing_ids:
    print('First missing video_ids:', missing_ids[:20])

# Nur Videos mit vorhandenen Frames weiterverarbeiten
df = df[df['has_frames']].reset_index(drop=True)
print(f'Restricted to {len(df)} rows with available frames')


Total videos in CSV: 500
Videos with frame folder: 500
Videos missing frame folder: 0
Restricted to 500 rows with available frames


In [ ]:
def compute_saturation(image_path):
    """Berechnet die Farbsättigung als Mittelwert des HSV-S-Kanals (0-255)."""
    try:
        img = cv2.imread(str(image_path))
        if img is None:
            return np.nan
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        saturation = float(np.mean(hsv[:, :, 1]))
        return saturation
    except Exception as e:
        print(f'Error reading {image_path}: {e}')
        return np.nan

def sample_frame_paths(video_id: str, duration_seconds=None, default_max_frames: int = DEFAULT_MAX_FRAMES_PER_VIDEO):
    folder = FRAME_ROOT / video_id
    if not folder.is_dir():
        return []

    frame_files = sorted(folder.glob('*.jpeg'))
    if not frame_files:
        return []

    # Ziel: ein Frame pro Videosekunde
    try:
        duration_value = float(duration_seconds)
    except (TypeError, ValueError):
        duration_value = np.nan

    if pd.notna(duration_value) and duration_value > 0:
        target_frames = int(np.ceil(duration_value))
    else:
        target_frames = default_max_frames

    target_frames = max(1, min(target_frames, len(frame_files)))
    step = max(1, len(frame_files) // target_frames)
    return frame_files[::step][:target_frames]



In [6]:
# Alle Videos verarbeiten
saturation_values = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    video_id = str(row.get('id') or row.get('video_id'))
    frames = sample_frame_paths(video_id)
    if not frames:
        saturation_values.append(np.nan)
        continue

    sat_list = []
    for fp in frames:
        sat = compute_saturation(fp)
        if not np.isnan(sat):
            sat_list.append(sat)

    mean_saturation = float(np.mean(sat_list)) if sat_list else np.nan
    saturation_values.append(mean_saturation)

# Ergebnis-Spalten ergänzen und CSV speichern
df['saturation_index'] = saturation_values
df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved results to {OUTPUT_CSV}')


100%|██████████| 500/500 [01:21<00:00,  6.13it/s]


Saved results to ..\..\data\04_analysis_results\visual_features\02_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_saturation.csv


In [7]:
df[['influencer_type', 'saturation_index']].groupby('influencer_type').describe()


saturation_index                                              \
                           count       mean        std        min        25%   
influencer_type                                                                
ai                         250.0  83.861156  37.431045  24.461095  56.986638   
real                       250.0  66.768843  19.400877  29.933247  51.578451   

                                                    
                       50%         75%         max  
influencer_type                                     
ai               77.488857  100.232239  225.815744  
real             65.029545   79.122935  145.145380